# Reproducing the Factorizer Paper — Using Authors' Official Library

**Paper:** Ashtari et al., *Factorizer: A Scalable Interpretable Approach to Context Modeling for Medical Image Segmentation*  
**Journal:** Medical Image Analysis, Vol. 84, 2023  
**arXiv:** https://arxiv.org/abs/2202.12295  
**DOI:** https://doi.org/10.1016/j.media.2022.102706  

---

This notebook reproduces the paper using the **authors' own official library code** from `factorizer_official/`.  
Our own reimplementation (`model.py`) is in the separate notebook `factorizer_reproduction.ipynb` and is **not touched here**.

### Sections
1. Setup — import official library from cloned repo
2. NMF Layer — official HALS implementation
3. Shifted Window Matricize — official implementation
4. FactorizerBlock — official
5. Full Swin Factorizer — parameter count verification (paper: 5.9M)
6. Load real BraTS data
7. Forward pass on BraTS patch
8. Training loop — demo on 10 cases
9. Evaluation — Dice ET / TC / WT
10. Paper results reference table (Table 1)

---
## Section 1 — Setup

In [ ]:
print(f'Notebook version: FIXED-LOSS-v2 | loss=binary_sigmoid_brats | ET_pos_weight=10')

In [ ]:
import os, sys, json, warnings, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.nn.functional as torch_F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
warnings.filterwarnings('ignore')
NOTEBOOK_DIR = '/kaggle/working'
OFFICIAL_DIR = '/kaggle/input/datasets/amrit9326/code-factorizer/factorizer_official'
sys.path.insert(0, OFFICIAL_DIR)
import factorizer as ft
REPO_DIR = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR = '/kaggle/input/datasets/amrit9326/task01-braintumour/Task01_BrainTumour'
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Official library: {ft.__file__}')
print(f'PyTorch:  {torch.__version__}')
print(f'Device:   {device}')
print(f'BraTS:    {DATA_DIR}')
print(f'Data exists: {os.path.exists(DATA_DIR)}')

---
## Section 2 — NMF Layer (Official)

The official library implements HALS as `CoordinateDescent` with `project=ReLU`.  
This is **true rank-by-rank coordinate descent** matching Algorithm 2 of the paper exactly:

```
For each rank r (keeping all other ranks fixed):
  f_r ← max(0,  (X g_r  −  Σ_{s≠r} f_s g_s^T g_r)  /  ‖g_r‖²  )
  g_r ← max(0,  (X^T f_r  −  Σ_{s≠r} g_s f_s^T f_r)  /  ‖f_r‖²  )
```
Never inverts a matrix → never singular.

In [ ]:
nmf = ft.NMF(size=(8, 512), rank=1, num_iters=5, init='uniform', solver='hals')
print(f'Solver:      {nmf.solver.__class__.__name__}')
print(f'Init:        {nmf.init.__class__.__name__}')
print(f'Rank:        {nmf.rank}')
print(f'Iterations:  {nmf.num_iters}')
print(f'Grad steps:  {nmf.num_grad_steps}  (last N iters get gradients)')
print()
torch.manual_seed(42)
x = torch.rand(1, 8, 512)
y = nmf(x)
print(f'Input:  {x.shape}  range [{x.min():.3f}, {x.max():.3f}]')
print(f'Output: {y.shape}  range [{y.min():.3f}, {y.max():.3f}]')
print(f'Reconstruction MSE: {((x - y) ** 2).mean():.4f}')
x2 = torch.rand(1, 8, 512, requires_grad=True)
ft.NMF(size=(8, 512), rank=1, num_iters=5)(x2).mean().backward()
print(f'Gradient norm: {x2.grad.norm():.4f}  ✓ differentiable')

---
## Section 3 — Shifted Window Matricize (Official)

Reshapes a 3D volume into a batch of matrices for NMF.  
`inverse_forward` reconstructs the original volume exactly — lossless.

In [ ]:
sw = ft.SWMatricize((None, 32, 64, 64, 64), head_dim=8, patch_size=8)
x_vol = torch.rand(1, 32, 64, 64, 64)
x_mat = sw(x_vol)
x_rec = sw.inverse_forward(x_mat)
print(f'Volume:       {x_vol.shape}  (B, C, H, W, D)')
print(f'Matricized:   {x_mat.shape}')
print(f'Reconstructed:{x_rec.shape}')
print(f'Perfect reconstruction: {torch.allclose(x_vol, x_rec, atol=1e-05)}')
print()
print('Matricized shape breakdown:')
print(f'  dim 0 = {x_mat.shape[0]} = B×n_heads×n_windows×2 (regular+shifted)')
print(f'  dim 1 = {x_mat.shape[1]} = head_dim E=8')
print(f'  dim 2 = {x_mat.shape[2]} = patch_size³ = 8³=512 spatial positions')

---
## Section 4 — FactorizerBlock (Official)

Structure (from Figure 2 of paper):
```
x → LayerNorm → PointwiseConv → SWMatricize → ReLU → NMF(HALS)
  → Dematricize → PointwiseConv → + x  (residual)
  → LayerNorm → MLP(GELU) → + x  (residual)
```

In [ ]:
blk = ft.FactorizerBlock(channels=32, spatial_size=(64, 64, 64), norm=ft.LayerNorm, reshape=(ft.SWMatricize, {'head_dim': 8, 'patch_size': 8}), act=nn.ReLU, factorize=ft.NMF, rank=1, num_iters=5, init='uniform', solver='hals', mlp_ratio=2, dropout=0.0)
n_params = sum((p.numel() for p in blk.parameters()))
x = torch.rand(1, 32, 64, 64, 64)
with torch.no_grad():
    y = blk(x)
print(f'FactorizerBlock params: {n_params:,}')
print(f'Input:  {x.shape}')
print(f'Output: {y.shape}  ← same shape (residual connection)')

---
## Section 5 — Full Swin Factorizer (Official)

Building the complete model with **paper-exact settings** (Section 4.2).  
Target: **5.9M parameters** as reported in Table 1.

In [ ]:
official_model = ft.Factorizer(in_channels=4, out_channels=3, spatial_size=(128, 128, 128), encoder_depth=(1, 1, 1, 1, 1), encoder_width=(32, 64, 128, 256, 512), strides=(1, 2, 2, 2, 2), decoder_depth=(1, 1, 1, 1), norm=ft.LayerNorm, reshape=(ft.SWMatricize, {'head_dim': 8, 'patch_size': 8}), act=nn.ReLU, factorize=ft.NMF, rank=1, num_iters=5, init='uniform', solver='hals', mlp_ratio=2, dropout=0.0, num_deep_supr=3).to(device)
n_params = sum((p.numel() for p in official_model.parameters() if p.requires_grad))
print(f'Official Swin Factorizer')
print(f'  Parameters: {n_params:,} ({n_params / 1000000.0:.2f}M)')
print(f'  Paper:      ~5.9M')
print(f'  Match:      {abs(n_params / 1000000.0 - 5.9) < 0.5}')
print()
stage_params = {}
for name, param in official_model.named_parameters():
    stage = name.split('.')[0]
    stage_params[stage] = stage_params.get(stage, 0) + param.numel()
print('Parameters per stage:')
for stage, count in stage_params.items():
    print(f'  {stage:12s}: {count:>8,}')

---
## Section 6 — Load Real BraTS Data

In [ ]:
with open(os.path.join(DATA_DIR, 'dataset.json')) as f:
    meta = json.load(f)
print(f"BraTS cases: {len(meta['training'])}")

def resolve_path(data_dir, rel_path):
    p = os.path.join(data_dir, rel_path.lstrip('./'))
    if not os.path.exists(p) and p.endswith('.nii.gz'):
        p = p[:-3]
    return p

def load_case(img_path, lbl_path):
    img = nib.load(img_path).get_fdata().astype(np.float32)
    lbl = nib.load(lbl_path).get_fdata().astype(np.int64)
    img_n = np.zeros_like(img)
    for c in range(4):
        ch = img[..., c]
        m = ch > 0
        if m.sum() > 0:
            img_n[..., c] = (ch - ch[m].mean()) / (ch[m].std() + 1e-08)
    return (torch.from_numpy(img_n.transpose(3, 0, 1, 2)), torch.from_numpy(lbl))

def crop_patch(img, lbl, size=(128, 128, 128)):
    ph, pw, pd = size
    C, H, W, D = img.shape
    if lbl.sum() > 0 and torch.rand(1).item() > 0.5:
        tv = torch.nonzero(lbl > 0)
        ch, cw, cd = tv[torch.randint(0, len(tv), (1,)).item()].tolist()
        sh = min(max(ch - ph // 2, 0), H - ph)
        sw = min(max(cw - pw // 2, 0), W - pw)
        sd = min(max(cd - pd // 2, 0), D - pd)
    else:
        sh = torch.randint(0, max(H - ph, 1), (1,)).item()
        sw = torch.randint(0, max(W - pw, 1), (1,)).item()
        sd = torch.randint(0, max(D - pd, 1), (1,)).item()
    return (img[:, sh:sh + ph, sw:sw + pw, sd:sd + pd], lbl[sh:sh + ph, sw:sw + pw, sd:sd + pd])
c0 = meta['training'][0]
ip = resolve_path(DATA_DIR, c0['image'])
lp = resolve_path(DATA_DIR, c0['label'])
print(f'Resolved image: {ip}')
print(f'Resolved label: {lp}')
img_t, lbl_t = load_case(ip, lp)
print(f'Full volume: {img_t.shape}  Labels: {lbl_t.unique().tolist()}')

---
## Section 7 — Forward Pass on BraTS Patch

In [ ]:
img_c, lbl_c = crop_patch(img_t, lbl_t, (128, 128, 128))
x_in = img_c.unsqueeze(0).to(device)
official_model.eval()
with torch.no_grad():
    outputs = official_model(x_in)
print(f'Input: {x_in.shape}')
if isinstance(outputs, list):
    for i, o in enumerate(outputs):
        print(f'Output {i} (deep supervision): {o.shape}')
    main_out = outputs[0]
else:
    print(f'Output: {outputs.shape}')
    main_out = outputs
et = (torch.sigmoid(main_out[:, 0]) > 0.5).squeeze(0).cpu()
tc = (torch.sigmoid(main_out[:, 1]) > 0.5).squeeze(0).cpu()
wt = (torch.sigmoid(main_out[:, 2]) > 0.5).squeeze(0).cpu()
et = et & tc & wt
tc = tc & wt
print(f'\nET voxels: {et.sum().item()}  TC voxels: {tc.sum().item()}  WT voxels: {wt.sum().item()}')
print(f'Forward pass on real BraTS data: OK ✓')

---
## Section 8 — Full Training Loop (Kaggle)

Paper setup (Section 4.2):
- AdamW, lr=1e-4, wd=1e-2
- Linear warmup 2000 steps → cosine annealing to 100k steps
- Dice + CE loss, deep supervision weights 1.0 / 0.5 / 0.25
- Checkpoint saved every 1000 steps → survives Kaggle session timeout
- Resume from checkpoint if one exists

In [ ]:
def binary_dice(logit, target, smooth=1e-05):
    p = torch.sigmoid(logit)
    inter = (p * target).sum()
    return 1.0 - (2 * inter + smooth) / (p.sum() + target.sum() + smooth)

def brats_loss(logits, target):
    gt_et = (target == 3).float()
    gt_tc = ((target == 1) | (target == 3)).float()
    gt_wt = (target >= 1).float()
    et_pos_weight = torch.tensor([10.0], device=logits.device)
    tc_pos_weight = torch.tensor([3.0], device=logits.device)
    wt_pos_weight = torch.tensor([2.0], device=logits.device)
    loss_et = binary_dice(logits[:, 0], gt_et) + torch.nn.functional.binary_cross_entropy_with_logits(logits[:, 0], gt_et, pos_weight=et_pos_weight)
    loss_tc = binary_dice(logits[:, 1], gt_tc) + torch.nn.functional.binary_cross_entropy_with_logits(logits[:, 1], gt_tc, pos_weight=tc_pos_weight)
    loss_wt = binary_dice(logits[:, 2], gt_wt) + torch.nn.functional.binary_cross_entropy_with_logits(logits[:, 2], gt_wt, pos_weight=wt_pos_weight)
    return loss_et + loss_tc + loss_wt

def deep_supervision_loss(outputs, target, weights=(1.0, 0.5, 0.25)):
    total = 0.0
    for out, w in zip(outputs, weights):
        if out.shape[2:] != target.shape[1:]:
            t = torch_F.interpolate(target.float().unsqueeze(1), size=out.shape[2:], mode='nearest').squeeze(1).long()
        else:
            t = target
        total += w * brats_loss(out, t)
    return total
print('Loss functions defined.')
dummy_out = [torch.randn(1, 3, 32, 32, 32), torch.randn(1, 3, 16, 16, 16), torch.randn(1, 3, 8, 8, 8)]
dummy_lbl = torch.randint(0, 4, (1, 32, 32, 32))
loss = deep_supervision_loss(dummy_out, dummy_lbl)
print(f'Loss test: {loss.item():.4f} ✓')
print('ET pos_weight=10 | TC pos_weight=3 | WT pos_weight=2')

In [ ]:
class BraTSDataset(Dataset):

    def __init__(self, cases, data_dir, patch_size=(128, 128, 128), augment=True):
        self.cases = cases
        self.data_dir = data_dir
        self.patch_size = patch_size
        self.augment = augment

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        c = self.cases[idx]
        ip = resolve_path(self.data_dir, c['image'])
        lp = resolve_path(self.data_dir, c['label'])
        img, lbl = load_case(ip, lp)
        img, lbl = crop_patch(img, lbl, self.patch_size)
        if self.augment:
            for ax in range(1, 4):
                if torch.rand(1).item() > 0.5:
                    img = torch.flip(img, [ax])
                    lbl = torch.flip(lbl, [ax - 1])
        return {'image': img, 'label': lbl}
all_cases = meta['training']
n_val = max(1, int(len(all_cases) * 0.1))
train_cases = all_cases[n_val:]
val_cases = all_cases[:n_val]
train_ds = BraTSDataset(train_cases, DATA_DIR, augment=True)
val_ds = BraTSDataset(val_cases, DATA_DIR, augment=False)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} cases | Val: {len(val_ds)} cases')

In [ ]:
import contextlib
CHECKPOINT_DIR = '/kaggle/working'
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'checkpoint.pt')
EVAL_LOG_PATH = os.path.join(CHECKPOINT_DIR, 'eval_log.json')
RESUME_PATH = '/kaggle/input/datasets/amrit9326/checkpoint-new/checkpoint.pt'
N_STEPS = 100000
SAVE_EVERY = 500
PRINT_EVERY = 100
train_model = ft.Factorizer(in_channels=4, out_channels=3, spatial_size=(128, 128, 128), encoder_depth=(1, 1, 1, 1, 1), encoder_width=(32, 64, 128, 256, 512), strides=(1, 2, 2, 2, 2), decoder_depth=(1, 1, 1, 1), norm=ft.LayerNorm, reshape=(ft.SWMatricize, {'head_dim': 8, 'patch_size': 8}), act=nn.ReLU, factorize=ft.NMF, rank=1, num_iters=5, init='uniform', solver='hals', mlp_ratio=2, dropout=0.0, num_deep_supr=3).to(device)
n = sum((p.numel() for p in train_model.parameters()))
print(f'Model params: {n:,} ({n / 1000000.0:.2f}M)  — paper: 5.9M')
optimizer = optim.AdamW(train_model.parameters(), lr=0.0001, weight_decay=0.01)
warmup = LinearLR(optimizer, start_factor=0.1, total_iters=2000)
cosine = CosineAnnealingLR(optimizer, T_max=N_STEPS - 2000, eta_min=1e-06)
sched = SequentialLR(optimizer, [warmup, cosine], milestones=[2000])
loss_history = []
eval_log = []
step = 0
resume_file = RESUME_PATH or (CHECKPOINT_PATH if os.path.exists(CHECKPOINT_PATH) else None)
if resume_file and os.path.exists(resume_file):
    print(f'Resuming from {resume_file}')
    ckpt = torch.load(resume_file, map_location=device, weights_only=False)
    train_model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    sched.load_state_dict(ckpt['scheduler'])
    step = ckpt['step']
    loss_history = ckpt.get('loss_history', [])
    eval_log = ckpt.get('eval_log', [])
    print(f'Resumed at step {step}')
else:
    print('Starting from scratch')

def run_eval(model, loader, current_step):
    model.eval()
    all_dice = []
    with torch.no_grad():
        for batch in loader:
            imgs = batch['image'].to(device)
            labels = batch['label'].cpu()
            outs = model(imgs)
            logits = outs[0]
            et = (torch.sigmoid(logits[:, 0]) > 0.5).cpu()
            tc = (torch.sigmoid(logits[:, 1]) > 0.5).cpu()
            wt = (torch.sigmoid(logits[:, 2]) > 0.5).cpu()
            et = et & tc & wt
            tc = tc & wt
            for b in range(imgs.shape[0]):
                gt = labels[b]
                gt_et = (gt == 3).float()
                gt_tc = ((gt == 1) | (gt == 3)).float()
                gt_wt = (gt >= 1).float()
                res = {}
                for name, p, g in [('ET', et[b], gt_et), ('TC', tc[b], gt_tc), ('WT', wt[b], gt_wt)]:
                    p = p.float()
                    inter = (p.flatten() * g.flatten()).sum()
                    res[name] = (2 * inter / (p.sum() + g.sum() + 1e-05)).item() * 100
                res['mean'] = np.mean(list(res.values()))
                all_dice.append(res)
    avg = {k: np.mean([d[k] for d in all_dice]) for k in ['ET', 'TC', 'WT', 'mean']}
    avg['step'] = current_step
    model.train()
    return avg
train_model.train()
while step < N_STEPS:
    for batch in train_loader:
        if step >= N_STEPS:
            break
        imgs = batch['image'].to(device)
        labels = batch['label'].to(device)
        outputs = train_model(imgs)
        loss = deep_supervision_loss(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(train_model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
        sched.step()
        loss_history.append(loss.item())
        step += 1
        if step % PRINT_EVERY == 0:
            avg_loss = np.mean(loss_history[-PRINT_EVERY:])
            print(f'  Step {step:>6}/{N_STEPS} | Loss: {avg_loss:.4f} | LR: {sched.get_last_lr()[0]:.2e}')
        if step % SAVE_EVERY == 0:
            print(f'  Evaluating at step {step}...')
            avg = run_eval(train_model, val_loader, step)
            eval_log.append(avg)
            with open(EVAL_LOG_PATH, 'w') as f:
                json.dump(eval_log, f, indent=2)
            print(f"  Val  ET:{avg['ET']:.2f}%  TC:{avg['TC']:.2f}%  WT:{avg['WT']:.2f}%  Mean:{avg['mean']:.2f}%")
            ckpt = {'step': step, 'model': train_model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': sched.state_dict(), 'loss_history': loss_history, 'eval_log': eval_log}
            torch.save(ckpt, CHECKPOINT_PATH)
            print(f'  Checkpoint saved at step {step} → {CHECKPOINT_PATH}')
print(f'\nDone. Final loss: {loss_history[-1]:.4f}')

In [ ]:
plt.figure(figsize=(10, 3))
window = min(100, len(loss_history))
smoothed = np.convolve(loss_history, np.ones(window) / window, mode='valid')
plt.plot(smoothed, 'b-', linewidth=1.5, label=f'Loss (smoothed {window}-step avg)')
plt.xlabel('Step')
plt.ylabel('Loss (Dice + CE)')
plt.title(f'Training Loss — Factorizer (step {step}/{N_STEPS})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/fig_loss.png', dpi=120)
plt.show()
print(f'Steps completed: {step} / {N_STEPS}')

---
## Section 9 — Evaluation: Dice ET / TC / WT

In [ ]:
def brats_dice(pred_et, pred_tc, pred_wt, gt):
    gt_et = (gt == 3).float()
    gt_tc = ((gt == 1) | (gt == 3)).float()
    gt_wt = (gt >= 1).float()
    results = {}
    for name, p, g in [('ET', pred_et, gt_et), ('TC', pred_tc, gt_tc), ('WT', pred_wt, gt_wt)]:
        p = p.float()
        g = g.float()
        inter = (p.flatten() * g.flatten()).sum()
        results[name] = (2 * inter / (p.sum() + g.sum() + 1e-05)).item() * 100
    results['mean'] = np.mean(list(results.values()))
    return results
train_model.eval()
all_dice = []
with torch.no_grad():
    for batch in val_loader:
        imgs = batch['image'].to(device)
        labels = batch['label'].cpu()
        outs = train_model(imgs)
        logits = outs[0]
        et = (torch.sigmoid(logits[:, 0]) > 0.5).cpu()
        tc = (torch.sigmoid(logits[:, 1]) > 0.5).cpu()
        wt = (torch.sigmoid(logits[:, 2]) > 0.5).cpu()
        et = et & tc & wt
        tc = tc & wt
        for b in range(imgs.shape[0]):
            all_dice.append(brats_dice(et[b], tc[b], wt[b], labels[b]))
avg = {k: np.mean([d[k] for d in all_dice]) for k in ['ET', 'TC', 'WT', 'mean']}
print('Validation Results:')
print(f"  ET:   {avg['ET']:.2f}%")
print(f"  TC:   {avg['TC']:.2f}%")
print(f"  WT:   {avg['WT']:.2f}%")
print(f"  Mean: {avg['mean']:.2f}%")
print()
print('Paper target (100k steps, 484 cases, 5-fold CV):')
print('  ET: 79.33% | TC: 83.14% | WT: 90.16% | Mean: 84.21%')

---
## Section 10 — Paper Results Reference (Table 1)

In [ ]:
import pandas as pd
df = pd.DataFrame({'Model': ['nnU-Net', 'Res-U-Net', 'Performer', 'TransBTS', 'UNETR', 'Swin UNETR', 'nnFormer', 'Global Factorizer', 'Local Factorizer', 'Swin Factorizer ★'], 'Params (M)': [28.7, 28.9, 6.7, 33.0, 111.5, 62.2, 149.4, 5.9, 5.9, 5.9], 'FLOPs (G)': [1152.8, 1168.6, 222.7, 653.0, 1124.9, 1549.8, 489.9, 170.0, 170.0, 174.2], 'ET (%)': [75.89, 77.95, 78.43, 73.46, 76.4, 76.5, 77.71, 78.2, 78.67, 79.33], 'TC (%)': [81.94, 82.49, 82.34, 80.0, 82.06, 82.65, 83.41, 82.86, 82.85, 83.14], 'WT (%)': [90.09, 90.39, 88.72, 85.42, 89.37, 90.1, 90.05, 88.65, 89.3, 90.16], 'Mean (%)': [82.64, 83.61, 83.16, 79.63, 82.61, 83.08, 83.72, 83.24, 83.61, 84.21], 'HD95 (mm)': [9.19, 7.52, 10.21, 17.12, 9.56, 9.91, 6.93, 9.71, 7.41, 6.89]}).set_index('Model')

def hi(s):
    return ['background:#d4edda;font-weight:bold' if '★' in s.name else '' for _ in s]
display(df.style.apply(hi, axis=1).format(precision=2).set_caption('Table 1 — BraTS 2021, 5-fold CV (Ashtari et al. 2023)'))
print('\n★ = Target to reproduce using official library + full training')
print('  Run: python main.py (our training script, 100k steps)')
print('  Or:  use official library with MONAI training loop')